## TUGAS MANDIRI 

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

**Buat SparkSession**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession dibuat")

SparkSession dibuat


### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [5]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.



**Baca transaksi_tugas5.csv dari HDFS menjadi df_transaksi, tambahkan kolom pendapatan (unit_terjual x harga_satuan).**
**Buat df_target dari dictionary data_target_cabang di atas**

Kerjakan bagian A sampai D berikut:

In [9]:
df_transaksi = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv")

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



In [10]:
df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



**A. Join & Perbandingan Target**

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

In [19]:
ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan"))

hasil = ringkasan.join(df_target, on = "kota", how = "left")
hasil1 = hasil.withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100)
hasil1.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota**

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

In [25]:
#  1. Gunakan 'order_id' sebagai secondary sort agar hasil konsisten
window_spec = Window.partitionBy("kategori").orderBy(
    col("pendapatan").desc(), 
)

# 2. Tambahkan kolom rank menggunakan row_number()
df_ranked = df_transaksi.withColumn("rank_dalam_kategori", row_number().over(window_spec))

# 3. Filter 2 transaksi teratas dan tampilkan
df_ranked.filter(col("rank_dalam_kategori") <= 1) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

+--------+--------------------+----------+------------+------------+----------+-------------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|rank_dalam_kategori|
+--------+--------------------+----------+------------+------------+----------+-------------------+
|  TRX-35|          Elektronik| Purworejo|           9|      150000|   1350000|                  1|
|  TRX-92|             Fashion|      Solo|           9|      150000|   1350000|                  1|
| TRX-157|Kesehatan & Kecan...|Yogyakarta|           9|      150000|   1350000|                  1|
|  TRX-81|   Makanan & Minuman|  Semarang|           9|      150000|   1350000|                  1|
| TRX-350|        Rumah Tangga|Yogyakarta|           9|      150000|   1350000|                  1|
+--------+--------------------+----------+------------+------------+----------+-------------------+



**C. Spark SQL** 

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.


In [37]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'")

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'


In [39]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |   produk|       true|
|         |   target|       true|
|         |transaksi|       true|
+---------+---------+-----------+



In [40]:
hasil_sql = spark.sql('''
    SELECT 
        t.kota,
        ta.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target ta 
        ON t.kota = ta.kota
    GROUP BY 
        t.kota,
        ta.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

hasil_sql.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan**

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

Menurut Sari(saya sendiri) berdasarkan hasil bagian A dan B, cabang dengan kinerja paling baik adalah purwoerejo. Hal ini dikarenakan pencapain persen kota purworejo paling banyak yaitu 152.16%. Nah kalo cabang yang perlu perhatian manajemen lebih lanjut adalah cabang kota Semarang. Cabang ini memiliki capaian persen paling sedikit yaitu hanya sekitar 69% an saja. Dengan sedikitnya capaian persen dapat disimpulkan bahwa kota itu memang gak becus pic nya, harus di eval lagi. Intiya cabang kta semarang ini harus di audit dan diperlukan pelatihan-pelatihan yang lebih lanjut pada tim di cabang ini. Tak hanya itu, kota magelang juga perlu perhatian. Kenapa? pada tabel jumlah transaksi kota magelang hanya terjual 86. Oleh karena itu, mbak rani ini juga perlu audit dan disemangati supaya penjualan bisa meningat seperti di kota purworejo.